In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder
from RMSELoss import RMSELoss
import plotly.graph_objects as go

# Getting Dataframe

In [2]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)



# Encode Categorical_Features

In [3]:
# Define the categorical features
categorical_features = ["Material"]

le = LabelEncoder()

for feature in categorical_features:
    x_train[feature] = le.fit_transform(x_train[feature])
    x_dev[feature] = le.transform(x_dev[feature])  

# Split Categorical_Features

In [4]:
# Drop categorical features to get the continuous features
x_train_numerical_features = x_train.drop(categorical_features, axis=1)
x_dev_numerical_features = x_dev.drop(categorical_features, axis=1)

# Seperate the categorical features
x_train_categorical_features = x_train[categorical_features]
x_dev_categorical_features = x_dev[categorical_features]

# Change df into Tensors

In [5]:
train_tensor = torch.tensor(x_train.to_numpy(), dtype=torch.float)
x_train_numer_tensor = torch.tensor(x_train_numerical_features.to_numpy(),dtype=torch.float)
x_dev_numer_tensor = torch.tensor(x_dev_numerical_features.to_numpy(),dtype=torch.float)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float)

dev_tensor = torch.tensor(x_dev.to_numpy(), dtype=torch.float)
x_train_categorical_features_tensor = torch.tensor(x_train_categorical_features.to_numpy(),dtype=torch.long)
x_dev_categorical_features_tensor = torch.tensor(x_dev_categorical_features.to_numpy(),dtype=torch.long)
y_dev_tensor = torch.tensor(y_dev.to_numpy(), dtype=torch.float)

# Create TensorDatasets for training and validation
train_ds = TensorDataset(
    x_train_categorical_features_tensor,
    x_train_numer_tensor,
    y_train_tensor
)
val_ds = TensorDataset(
    x_dev_categorical_features_tensor,
    x_dev_numer_tensor,
    y_dev_tensor
)
g = torch.Generator()
g.manual_seed(42)

# Create DataLoaders for training and validation
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator= g)
val_loader   = DataLoader(val_ds,   batch_size=70)



In [47]:
print(val_loader.batch_size)

70


# Check Tensors


In [6]:
#torch.set_printoptions(sci_mode=False, precision=3)
#print(x_train_numer_tensor)

# Define Model

In [8]:
# categories is defined as a tuple, we only have one categorical feature "Material"
# the second parameter has to be empty for the model to work correctly
# in hard numbers this is displaying (1,)
model = FTTransformer(
    categories=(x_train_categorical_features.shape[1],),
    num_continuous=x_train_numerical_features.shape[1],
    dim=9,
    dim_out=1,
    depth=4,
    heads=4,
    attn_dropout=0.1,
    ff_dropout=0.1
)

# (Alternative) Initiate a saved Model

In [ ]:
#to be able to match the model, define the same parameters above in Define Model
#for example if the saved model has a dim of 9 the dim in define model has to be 9 as well

#saved models are in the folder "trained models"

#name of model
model_name = "model_lr0.00025_8_1_4_4_.3_.3Residual_batch70_long_epoch7100"

#path to the saved model
path = f'trained_models/physical_error_training/{model_name}.pt'
model.load_state_dict(torch.load(path))

<All keys matched successfully>

# Physical Epoch


In [9]:

#["Thickness A (mm)"] = 5
#["Thickness B (mm)"] = 6
# idxA = 5 , idxB = 6

idxA, idxB = 5, 6  

def physics_pull_force(x_cont):
    # x_cont: [B, num_cont_features]
    A = x_cont[:, idxA]               
    B = x_cont[:, idxB]               

    # element‐wise minimum of each row
    t = torch.minimum(A, B)           
    # f_phys = (π/4)*(4*√t)^2*(0.8*365)
    return ((torch.pi / 4) * (4 * torch.sqrt(t)).pow(2) * (0.8 * 365))


def train_one_epoch(train_loader):
    total_error     = 0.0


    for x_cat, x_cont, y in train_loader:
        optimizer.zero_grad()

        # ensure y is [B,1]
        y = y.unsqueeze(-1)




        # physics target (no grads needed for f_phys)
        with torch.no_grad():
            f_phys = physics_pull_force(x_cont).unsqueeze(-1)    # [B]
            
        # difference between target and caculated physics‐based target
        True_Error = y - f_phys

        # forward
        pred = model(x_cat, x_cont)   

        # error loss
        error_res_loss = criterion(pred, True_Error)

        # combined loss
        loss = error_res_loss
        loss.backward()
        optimizer.step()

        total_error += loss.item()



    return total_error / len(train_loader)




# Physical Error Training

In [10]:
from datetime import datetime

criterion = RMSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00025)

#Changeable Parameters
# ----------------------------------------------------#
#Model description for saving
model_description    = f'26FTTransformer_lr{optimizer.param_groups[0]["lr"]}_{model.to_logits[2].in_features}_{model.to_logits[2].out_features}_{len(model.transformer.layers)}_{model.transformer.layers[0][0].branch.heads}_{model.transformer.layers[0][0].branch.dropout.p}_{model.transformer.layers[0][1].branch[3].p}_Residualbatch{val_loader.batch_size}'
#_{model.dim_out}_{model.depth}_{model.heads}_{model.attn_dropout}_{model.ff_dropout}Errorbatch70
# Number of epochs to train
EPOCHS = 20000
#-----------------------------------------------------#

# Lists to store per‐epoch losses
train_phys_losses       = []
val_phys_losses         = []
prediction              = []
physics                 = []  
datasety                = []  


best_vloss   = float('inf')
last_ckpt  = None
#timestamp for manual checkpoint selection
timestamp    = datetime.now().strftime('%Y%m%d_%H%M%S')



for epoch in range(EPOCHS):
    print(f'\nEPOCH {epoch+1}/{EPOCHS}')

    # --------------------
    # 1) TRAINING PHASE
    # --------------------
    model.train()
    train_data = train_one_epoch(train_loader)

    train_phys_losses.append(train_data)
    print(f'train loss: {train_data:.4f}')

    
    # --------------------
    # 2) VALIDATION PHASE
    # --------------------
    model.eval()
    
    val_loss = 0.0
    with torch.no_grad():
        for x_cat, x_cont, y in val_loader:
            y = y.unsqueeze(-1)
            pred = model(x_cat, x_cont)

            f_phys = physics_pull_force(x_cont)    # [B]
            f_phys = f_phys.unsqueeze(-1) 


            True_Error = y - f_phys

            # 1) loss
            val_loss += criterion(f_phys+ pred, f_phys + True_Error).item()

    avg_vloss = val_loss / len(val_loader)


    val_phys_losses.append(avg_vloss)
    prediction.append(f_phys + pred)
    physics.append(f_phys)
    datasety.append(y)


    print(f'valid loss: {avg_vloss:.4f}')
    

    # --------------------
    # 3) CHECKPOINTING
    # --------------------
    # to sort through the checkpoints manually, add timestamp to the filename
    # remove avg_total < best_vloss constraint and 2nd if condition

    #_date_{timestamp}
    if (epoch + 1) % 100 == 0 and avg_vloss < best_vloss:
        if last_ckpt is not None:
            os.remove(last_ckpt)

        best_vloss = avg_vloss
        ckpt_path = f'trained_models/physical_error_training/model_{model_description}_epoch{epoch+1}.pt'
        torch.save(model.state_dict(), ckpt_path)
        last_ckpt = ckpt_path



EPOCH 1/20000
train loss: 704.1358
valid loss: 745.7030

EPOCH 2/20000
train loss: 706.9076
valid loss: 745.6387

EPOCH 3/20000
train loss: 700.7524
valid loss: 745.6064

EPOCH 4/20000
train loss: 702.5492
valid loss: 745.5807

EPOCH 5/20000
train loss: 702.6977
valid loss: 745.5601

EPOCH 6/20000
train loss: 702.8397
valid loss: 745.5425

EPOCH 7/20000
train loss: 705.3972
valid loss: 745.5280

EPOCH 8/20000
train loss: 702.0572
valid loss: 745.5151

EPOCH 9/20000
train loss: 714.8476
valid loss: 745.5033

EPOCH 10/20000
train loss: 702.7511
valid loss: 745.4919

EPOCH 11/20000
train loss: 708.4644
valid loss: 745.4804

EPOCH 12/20000
train loss: 703.8269
valid loss: 745.4690

EPOCH 13/20000
train loss: 702.3523
valid loss: 745.4576

EPOCH 14/20000
train loss: 709.0270
valid loss: 745.4463

EPOCH 15/20000
train loss: 705.3646
valid loss: 745.4349

EPOCH 16/20000
train loss: 703.7458
valid loss: 745.4236

EPOCH 17/20000
train loss: 707.7134
valid loss: 745.4122

EPOCH 18/20000
train l

# Plotly

In [24]:
epochs = list(range(1, len(train_phys_losses) + 1))

fig = go.Figure()

# Training loss trace
fig.add_trace(go.Scatter(
    x=epochs,
    y=train_phys_losses,
    mode="markers",
    name="Train Loss",
    marker=dict(color="blue", size=1)
))

# Validation loss trace
fig.add_trace(go.Scatter(
    x=epochs,
    y=val_phys_losses,
    mode="markers",
    name="Validation Loss",
    marker=dict(color="firebrick", size=1)
))

# Layout enhancements
fig.update_layout(
    title=f"Training & Validation Loss over {EPOCHS} Epochs - {model_description} - Loss on Physical Error",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(0, 50500, 2500)),  # Show ticks every 100 epochs
        tickfont=dict(size=10)
    ),
    yaxis=dict(
        tickformat=".2e" if max(train_phys_losses + val_phys_losses) > 1e4 else ".4f",  # Dynamic formatting
        gridcolor="lightgray"
    ),
    legend=dict(
        x=0.7,        # push legend past the right edge
        y=1,
        xanchor="left",
        borderwidth=1
    ),
    
    template="plotly_white",
    margin=dict(t=60, b=40)
)

fig.show()


In [23]:
# After training, do a one‐pass over val_loader:
all_true, all_phys, all_pred = [], [], []

with torch.no_grad():
    for x_cat, x_cont, y in val_loader:
        y   = y.unsqueeze(-1)                             # [B,1]
        f_p = physics_pull_force(x_cont).unsqueeze(-1)     # [B,1]
        r_p = model(x_cat, x_cont)                         # [B,1]
        y_p = (f_p + r_p)                                  # [B,1]

        all_true.append(y.ravel())
        all_phys.append(f_p.ravel())
        all_pred.append(y_p.ravel())

# flatten
all_true = np.concatenate(all_true)
all_phys = np.concatenate(all_phys)
all_pred = np.concatenate(all_pred)

#plot per‐sample
sample = np.arange(len(all_true))

fig = go.Figure()
fig.add_trace(go.Scatter(x=sample, y=all_true, mode="markers",
                         name="True Data", marker=dict(color="black", size=6)))
fig.add_trace(go.Scatter(x=sample, y=all_phys, mode="markers",
                         name="Physics Only", marker=dict(color="firebrick", size=6)))
fig.add_trace(go.Scatter(x=sample, y=all_pred, mode="markers",
                         name="Prediction", marker=dict(color="blue", size=6)))

fig.update_layout(
    title=f"All Validation Samples: True vs Physicsvs Prediction Epochs {EPOCHS} - {model_description}",
    xaxis_title="Sample Index",
    yaxis_title="Pull-Force",
    template="seaborn"
)
fig.show()


# Check Validation Loss and R2

In [17]:
# Collect predictions and true values from the validation set
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for x_cat, x_cont, y in val_loader:
        y = y.unsqueeze(-1)               
        all_targets.append(y)

        # 1) Physics pull‐force 
        F_phys = physics_pull_force(x_cont).unsqueeze(-1)

        # 2) Error
        True_Error = model(x_cat, x_cont)

        # 3) full prediction
        F_pred = (F_phys + True_Error)
        all_preds.append(F_pred)

# Concatenate batches
y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
R2   = r2_score(y_true, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")

MAE:  166.40
RMSE: 317.78
R2: 0.56


# Plotly(old representation)

In [18]:
import plotly.graph_objects as go

fig = go.Figure()

# Training loss trace
fig.add_trace(go.Scatter(
    y=train_data_losses,
    mode="lines+markers",
    name="Train Data Loss",
    line=dict(color="lightblue", width=2),
    marker=dict(size=4)
))

# Validation loss trace
fig.add_trace(go.Scatter(
    y=val_data_losses,
    mode="lines+markers",
    name="Validation Data Loss",
    line=dict(color="orange", width=2),
    marker=dict(size=4)
))

# Layout
fig.update_layout(
    title="Training & Validation Loss over Epochs  - lr0.00025_8_1_4_4_.3_.3 - Physics Lambda 4",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(0, 10500, 500)),  # Show ticks every 100 epochs
        tickfont=dict(size=10)
    ),
    template="plotly_white",
    legend=dict(
        x=0.7,        # push legend past the right edge
        y=1,
        xanchor="left",
        borderwidth=1
    ),
    margin=dict(t=60, b=40)  # give extra room on right for the legend
)

fig.show()
